# Module 6: Hybrid search + RRF reranking (no Elasticsearch)

The course runs these lessons on Elasticsearch (Docker). Docker is not installed here, so we build the same ideas from what we already have:

- **keyword search**: minsearch `Index` with the boosts tuned in Module 4 (question 1, answer 2, section 0.1)
- **vector search**: ONNX `Embedder` (all-MiniLM-L6-v2) + minsearch `VectorSearch` from Module 2
- **hybrid**: Reciprocal Rank Fusion (RRF) of both ranked lists (course lesson 02 code)

Evaluation: Module 4 Hit Rate / MRR on the course ground-truth questions whose document is still in today's FAQ.

Kernel / run: uses the `02-vector-search/.venv` (Python 3.14, onnxruntime).

## Data

In [1]:
import pandas as pd
from ingest import load_faq_data

documents = [d for d in load_faq_data() if d["course"] == "llm-zoomcamp"]
doc_ids = {d["id"] for d in documents}

gt = pd.read_csv("../04-evaluation/data/course/ground_truth-new.csv").to_dict(orient="records")
ground_truth = [r for r in gt if r["document"] in doc_ids]
len(documents), len(ground_truth)

(161, 315)

## Keyword search (Module 1/4)

In [2]:
try:
    from minsearch import Index, VectorSearch
except ImportError:
    from minsearch_lite import Index
    VectorSearch = None

text_index = Index(text_fields=["question", "section", "answer"], keyword_fields=["course"])
text_index.fit(documents)

def keyword_search(query, num_results=5):
    return text_index.search(query, num_results=num_results,
                             boost_dict={"question": 1.0, "answer": 2.0, "section": 0.1})

## Vector search (Module 2)

In [3]:
import numpy as np, time
from embedder import Embedder

model = Embedder("../02-vector-search/models/Xenova/all-MiniLM-L6-v2")

t0 = time.time()
texts = [d["question"] + " " + d["answer"] for d in documents]
X = np.array([v for i in range(0, len(texts), 50) for v in model.encode_batch(texts[i:i + 50])])
print(f"embedded {X.shape} in {time.time() - t0:.1f} s")

vindex = VectorSearch(keyword_fields=["course"])
vindex.fit(X, documents)

def vector_search(query, num_results=5):
    return vindex.search(model.encode(query), num_results=num_results)

embedded (161, 384) in 6.5 s


## Reciprocal Rank Fusion (course lesson 02)
`score(doc) = sum over lists of 1 / (k + rank + 1)`. The course keys documents by `question`; we key by `id` (two FAQ entries can share a question text).

In [4]:
def rrf(search_results, k=1, num_results=10):
    scores = {}
    doc_map = {}
    for results in search_results:
        for rank, doc in enumerate(results):
            key = doc["id"]
            if key not in scores:
                scores[key] = 0
                doc_map[key] = doc
            scores[key] += 1 / (k + rank + 1)
    ranked = sorted(scores.items(), key=lambda x: x[1], reverse=True)
    return [doc_map[key] for key, _ in ranked[:num_results]]

# course worked example: text [A..E], vector [C,B,F,G,A], k=1 -> C first
A, B, C, D, E, F, G = ({"id": x} for x in "ABCDEFG")
[d["id"] for d in rrf([[A, B, C, D, E], [C, B, F, G, A]], k=1)]

['C', 'A', 'B', 'F', 'D', 'G', 'E']

In [5]:
def hybrid_search(query, k=60, num_results=5, depth=10):
    kw = keyword_search(query, num_results=depth)
    vec = vector_search(query, num_results=depth)
    return rrf([kw, vec], k=k, num_results=num_results)

## Evaluation (Module 4 metrics)

In [6]:
from tqdm.auto import tqdm

def hit_rate(relevance):
    return sum(1 in line for line in relevance) / len(relevance)

def mrr(relevance):
    total = 0.0
    for line in relevance:
        for rank, v in enumerate(line):
            if v == 1:
                total += 1 / (rank + 1)
                break
    return total / len(relevance)

def evaluate(ground_truth, search_function):
    relevance = []
    for q in tqdm(ground_truth):
        results = search_function(q["question"])
        relevance.append([int(d["id"] == q["document"]) for d in results])
    return {"hit_rate": round(hit_rate(relevance), 3), "mrr": round(mrr(relevance), 3)}

C:\Users\vasus\Projects\agent-ai-bootcamp\02-vector-search\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [7]:
results = {
    "keyword (tuned boosts)": evaluate(ground_truth, keyword_search),
    "vector (ONNX MiniLM)": evaluate(ground_truth, vector_search),
    "hybrid RRF k=60": evaluate(ground_truth, lambda q: hybrid_search(q, k=60)),
    "hybrid RRF k=1": evaluate(ground_truth, lambda q: hybrid_search(q, k=1)),
}
df_results = pd.DataFrame(results).T
df_results

  0%|          | 0/315 [00:00<?, ?it/s]

  2%|▏         | 5/315 [00:00<00:06, 47.11it/s]

  3%|▎         | 10/315 [00:00<00:07, 39.19it/s]

  5%|▍         | 15/315 [00:00<00:07, 39.49it/s]

  6%|▋         | 20/315 [00:00<00:06, 42.84it/s]

...

 98%|█████████▊| 310/315 [00:20<00:00, 16.09it/s]

 99%|█████████▉| 312/315 [00:21<00:00, 14.06it/s]

100%|█████████▉| 314/315 [00:21<00:00, 14.16it/s]

100%|██████████| 315/315 [00:21<00:00, 14.83it/s]

,hit_rate,mrr
keyword (tuned boosts),0.908,0.786
vector (ONNX MiniLM),0.943,0.865
hybrid RRF k=60,0.946,0.840
hybrid RRF k=1,0.959,0.843


## Where does hybrid help? (paraphrased question from Module 1/2)

In [8]:
q = "Is it possible to enroll late?"
for name, fn in [("keyword", keyword_search), ("vector", vector_search), ("hybrid", hybrid_search)]:
    print(f"{name:8s}:", [d["question"][:55] for d in fn(q)][:3])

keyword : ['Is it a group project?', "My dataset is copyrighted and I can't publish it. Can I", 'uv says Failed to hardlink files in Codespaces. Is it a']
vector  : ['I just discovered the course. Can I still join?', 'Course: I have registered for the LLM Zoomcamp. When ca', 'Certificate: Can I follow the course in a self-paced mo']
hybrid  : ['Can I submit homework after the deadline, or get a dead', 'Is it a group project?', 'I just discovered the course. Can I still join?']
